# Platoon Ratio & Arrival Type from hi-res controller data

This notebook demonstrates the **`platoon_ratio`** aggregation in the
[`atspm`](https://github.com/ShawnStrasser/atspm) Python package (added in v2.5.0, [PR #26](https://github.com/ShawnStrasser/atspm/pull/26)).

**What it computes** (HCM 7th ed., Chapter 19):

$$
R_p = \frac{P}{g/C}
$$

where $P$ is the proportion of vehicles arriving on green (`Percent_AOG` from the `arrival_on_green`
aggregation) and $g/C$ is the green ratio, estimated from the phase's green time inside each aggregation bin.

| Arrival type | Platoon ratio $R_p$ | Progression quality |
|---|---|---|
| 1 | ≤ 0.50 | Very poor (dense platoon arrives at start of red) |
| 2 | 0.51 – 0.85 | Unfavorable |
| 3 | 0.86 – 1.15 | Random arrivals |
| 4 | 1.16 – 1.50 | Favorable |
| 5 | 1.51 – 2.00 | Highly favorable |
| 6 | > 2.00 | Exceptional (dense platoon arrives at start of green) |

**Two ways to use it:** run sections 1–4 on the two-hour sample bundled with the package (no data needed),
or jump to **section 5** and upload your own hi-res export from ATSPM (GDOT/UDOT), MaxTime, Econolite, etc.
The upload section detects the column names for you.


## 1. Install the package

In [ ]:
%pip install -q "atspm>=2.5.0"
import atspm, duckdb, pandas as pd, numpy as np
from atspm import SignalDataProcessor, sample_data
print("atspm", getattr(atspm, "__version__", "(dev)"), "| duckdb", duckdb.__version__)

## 2. Look at the bundled hi-res sample

Standard Indiana/UDOT event codes used by the aggregation:

- `1` phase begin green, `8` phase begin yellow — used to build green intervals per phase
- `82` detector on — actuations, matched to phases through the detector config (`Function = 'Advance'`)

In [ ]:
raw = sample_data.data.df()
cfg = sample_data.config.df()

print(f"{len(raw):,} events from {raw.TimeStamp.min()} to {raw.TimeStamp.max()} | devices: {raw.DeviceId.unique()}")
display(raw.head())
print("Advance detectors (these drive arrival-on-green and platoon ratio):")
display(cfg[cfg.Function == "Advance"].sort_values("Phase"))

## 3. Run the aggregations

`platoon_ratio` depends on `arrival_on_green`; the processor orders them automatically via
`AGGREGATION_DEPENDENCIES`, but we list both here for clarity.

In [ ]:
def run_platoon_ratio(events, detector_config, bin_size=15, latency_offset_seconds=0, output_dir="atspm_output"):
    """Run arrival_on_green + platoon_ratio and return the platoon_ratio table as a DataFrame."""
    params = {
        "raw_data": events,                 # DataFrame or file path
        "detector_config": detector_config, # DataFrame or file path
        "bin_size": bin_size,               # minutes
        "output_dir": output_dir,
        "output_format": "csv",
        "output_to_separate_folders": False,
        "remove_incomplete": False,
        "verbose": 0,
        "aggregations": [
            {"name": "arrival_on_green", "params": {"latency_offset_seconds": latency_offset_seconds}},
            {"name": "platoon_ratio",    "params": {}},
        ],
    }
    with SignalDataProcessor(**params) as p:
        p.load(); p.aggregate()
        return p.conn.query("SELECT * FROM platoon_ratio ORDER BY DeviceId, Phase, TimeStamp").df()

rp = run_platoon_ratio(raw, cfg)
print(f"{len(rp)} phase-bins")
rp.head(12)

### Sanity checks and the right way to summarise

- `Green_Ratio` must be within (0, 1] and `Platoon_Ratio` must equal `Percent_AOG / Green_Ratio` row by row.
- `Arrival_Type` follows the HCM thresholds.
- **Never average `Platoon_Ratio` across bins.** It is a ratio of ratios; sum the four raw columns
  (`Green_Actuations`, `Total_Actuations`, `Green_Seconds`, bin seconds) and divide once.

In [ ]:
assert rp.Green_Ratio.between(0, 1, inclusive="right").all()
assert np.allclose(rp.Platoon_Ratio, rp.Percent_AOG / rp.Green_Ratio, rtol=1e-4)
edges = [-np.inf, 0.50, 0.85, 1.15, 1.50, 2.00, np.inf]
assert (pd.cut(rp.Platoon_Ratio, edges, labels=[1, 2, 3, 4, 5, 6]).astype(int) == rp.Arrival_Type).all()
print("all checks passed")

def summarise(rp, bin_size=15, by=("DeviceId", "Phase")):
    """Period platoon ratio computed from summed parts (correct), alongside the naive mean (biased)."""
    s = rp.groupby(list(by)).agg(bins=("TimeStamp", "size"),
                                 Total_Actuations=("Total_Actuations", "sum"),
                                 Green_Actuations=("Green_Actuations", "sum"),
                                 Green_Seconds=("Green_Seconds", "sum"),
                                 naive_mean_Rp=("Platoon_Ratio", "mean"))
    s["Percent_AOG"] = s.Green_Actuations / s.Total_Actuations
    s["Green_Ratio"] = s.Green_Seconds / (s.bins * bin_size * 60)
    s["Platoon_Ratio"] = s.Percent_AOG / s.Green_Ratio
    s["Arrival_Type"] = pd.cut(s.Platoon_Ratio, edges, labels=[1, 2, 3, 4, 5, 6]).astype(int)
    return s.round(3)

summarise(rp)

## 4. Why platoon ratio and not just arrival on green?

Percent arrival on green rewards phases that simply get a lot of green. Platoon ratio normalizes
by the green ratio, so a coordinated through phase (phase 2/6, big split) and a minor phase can
be compared on the same scale, and a value near 1 means arrivals are no better than random.

In [ ]:
import plotly.graph_objects as go

SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
AT_BANDS = [(0, .5, "1  very poor"), (.5, .85, "2  unfavorable"), (.85, 1.15, "3  random"),
            (1.15, 1.5, "4  favorable"), (1.5, 2, "5  highly favorable"), (2, 99, "6  exceptional")]
AT_TINT = ["rgba(11,11,11,0.10)", "rgba(11,11,11,0.06)", "rgba(11,11,11,0.02)",
           "rgba(27,175,122,0.06)", "rgba(27,175,122,0.11)", "rgba(27,175,122,0.16)"]

def plot_rp(rp, title="Platoon ratio per bin", bin_size=15, width=980, height=560):
    # Interactive platoon-ratio chart: one line per device/phase, HCM arrival-type bands, hover with the full bin.
    ymax = float(min(8, max(2.2, rp.Platoon_Ratio.max() * 1.08)))
    fig = go.Figure()
    for (lo, hi, lbl), tint in zip(AT_BANDS, AT_TINT):
        if lo >= ymax: break
        fig.add_hrect(y0=lo, y1=min(hi, ymax), fillcolor=tint, line_width=0, layer="below")
        fig.add_annotation(x=1.0, xref="paper", y=(lo + min(hi, ymax)) / 2, xanchor="left", showarrow=False,
                           text=f"<span style='color:#898781'>AT {lbl}</span>", font=dict(size=11), xshift=8)
    fig.add_hline(y=1.0, line=dict(color="#c3c2b7", width=1, dash="dot"))
    groups = list(rp.groupby(["DeviceId", "Phase"]))
    for i, ((dev, ph), d) in enumerate(groups):
        col = SERIES[i % len(SERIES)]
        name = f"Phase {ph}" if rp.DeviceId.nunique() == 1 else f"{dev} · phase {ph}"
        custom = d[["Percent_AOG", "Green_Ratio", "Total_Actuations", "Green_Actuations", "Arrival_Type"]].to_numpy()
        fig.add_trace(go.Scatter(
            x=d.TimeStamp, y=d.Platoon_Ratio, mode="lines+markers", name=name,
            line=dict(color=col, width=2, shape="spline", smoothing=0.5), marker=dict(size=6, color=col,
            line=dict(color="#fcfcfb", width=1.5)), customdata=custom,
            hovertemplate=("<b>%{fullData.name}</b>  %{x|%a %H:%M}<br>"
                           "Platoon ratio <b>%{y:.2f}</b>  (arrival type %{customdata[4]})<br>"
                           "Arrivals on green %{customdata[0]:.0%}  ·  green ratio %{customdata[1]:.0%}<br>"
                           "%{customdata[3]:.0f} of %{customdata[2]:.0f} vehicles arrived on green<extra></extra>")))
    # direct labels at the line ends, nudged apart when two series end at nearly the same value
    ends = sorted([(d.Platoon_Ratio.iloc[-1], d.TimeStamp.iloc[-1], f"Phase {ph}" if rp.DeviceId.nunique() == 1 else f"{dev} · phase {ph}", SERIES[i % len(SERIES)])
                   for i, ((dev, ph), d) in enumerate(groups)], key=lambda t: t[0])
    gap, placed = ymax * 0.045, []
    for y, x, name, col in ends:
        if placed and y - placed[-1] < gap: y = placed[-1] + gap
        placed.append(y)
        fig.add_annotation(x=x, y=y, text=name, xanchor="left", xshift=6, showarrow=False, font=dict(color=col, size=12))
    fig.update_layout(
        title=dict(text=f"<b>{title}</b><br><span style='font-size:12px;color:#52514e'>"
                        f"R<sub>p</sub> = arrivals on green ÷ green ratio, per {bin_size}-min bin. 1.0 = random arrivals; bands = HCM arrival type</span>",
                   x=0.01, xanchor="left"),
        template="plotly_white", autosize=False, width=width, height=height, margin=dict(l=60, r=150, t=110, b=40),
        paper_bgcolor="#fcfcfb", plot_bgcolor="#fcfcfb", hovermode="x unified",
        font=dict(family="system-ui, -apple-system, Segoe UI, Roboto, sans-serif", color="#0b0b0b"),
        legend=dict(orientation="h", y=1.0, yanchor="bottom", x=1.0, xanchor="right", font=dict(size=12)),
        xaxis=dict(title=None, showgrid=False, tickformat="%H:%M", rangeslider=dict(visible=True, thickness=0.06),
                   linecolor="#c3c2b7"),
        yaxis=dict(title="Platoon ratio", range=[0, ymax], gridcolor="#e1e0d9", zeroline=False, tickformat=".1f"))
    fig.show()

plot_rp(rp, "Bundled sample: platoon ratio by phase")
pd.crosstab(rp.Phase, rp.Arrival_Type).reindex(columns=range(1, 7), fill_value=0)

## 5. Try it on your own controller data

Upload a **hi-res event export**. Any of these schemas work; the columns are detected by name:

| Source | Signal | Time | Event | Parameter |
|---|---|---|---|---|
| GDOT / UDOT ATSPM raw export | `Signal Id` / `SignalID` | `Timestamp` | `Event Code` / `EventCode` | `Event Parameter` / `EventParam` |
| atspm package / MaxTime | `DeviceId` | `TimeStamp` | `EventId` | `Parameter` |
| Econolite / generic | `signal`, `id`, `device` … | `time`, `date` … | `event`, `code` … | `param`, `channel` … |

Then tell the notebook which detector channels are the **advance (setback) detectors** and which phase each serves.
Either upload a detector-config file (any file with a channel column, a phase column and a type/function column that
says "Advance") or type the mapping in the box below, e.g. `7:2, 21:6, 22:6`. In GDOT ATSPM this is on the signal's
configuration page under *Detectors* — the rows whose Detection Types include **Advanced Count**.


In [ ]:
import re, io, os, glob

def _pick(cols, *patterns):
    """Return the first column whose normalised name matches any pattern (regex, case-insensitive)."""
    norm = {c: re.sub(r"[^a-z0-9]", "", str(c).lower()) for c in cols}
    for pat in patterns:
        for c, n in norm.items():
            if re.fullmatch(pat, n):
                return c
    return None

def read_any(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet": return pd.read_parquet(path)
    if ext in (".xlsx", ".xls"): return pd.read_excel(path)
    return pd.read_csv(path, sep=None, engine="python")   # sniffs comma / tab / semicolon

def standardise_events(df):
    """Map any common hi-res schema onto TimeStamp, DeviceId, EventId, Parameter."""
    cols = list(df.columns)
    m = {
        "DeviceId":  _pick(cols, r"deviceid", r"signalid", r"signal", r"controllerid", r"intersectionid", r"id", r"device"),
        "TimeStamp": _pick(cols, r"timestamp", r"time", r"datetime", r"date", r"eventtime"),
        "EventId":   _pick(cols, r"eventid", r"eventcode", r"event", r"code", r"eventtype"),
        "Parameter": _pick(cols, r"parameter", r"eventparameter", r"eventparam", r"param", r"channel", r"value"),
    }
    missing = [k for k, v in m.items() if v is None]
    if missing:
        raise ValueError(f"Could not find columns for {missing}. Columns present: {cols}")
    out = df.rename(columns={v: k for k, v in m.items()})[["TimeStamp", "DeviceId", "EventId", "Parameter"]].copy()
    out["TimeStamp"] = pd.to_datetime(out["TimeStamp"], errors="coerce")
    out = out.dropna(subset=["TimeStamp"])
    for k in ("EventId", "Parameter"):
        out[k] = pd.to_numeric(out[k], errors="coerce").astype("Int64")
    out = out.dropna(subset=["EventId", "Parameter"]).astype({"EventId": "int64", "Parameter": "int64"})
    try:
        out["DeviceId"] = pd.to_numeric(out["DeviceId"]).astype("int64")
    except Exception:
        out["DeviceId"] = out["DeviceId"].astype(str)
    out = out.drop_duplicates().sort_values("TimeStamp").reset_index(drop=True)
    print("column mapping:", {k: v for k, v in m.items()})
    return out

def standardise_config(df, devices):
    """Map a detector-config file onto DeviceId, Phase, Parameter, Function (keeping Advance rows only)."""
    cols = list(df.columns)
    ch  = _pick(cols, r"parameter", r"detchannel", r"detectorchannel", r"channel", r"detector", r"detid", r"detectorid")
    ph  = _pick(cols, r"phase", r"protectedphase", r"phasenumber")
    fn  = _pick(cols, r"function", r"detectiontypes", r"detectiontype", r"type", r"movementtype")
    dev = _pick(cols, r"deviceid", r"signalid", r"signal", r"id")
    if ch is None or ph is None:
        raise ValueError(f"Need a channel column and a phase column; found {cols}")
    out = pd.DataFrame({"Parameter": pd.to_numeric(df[ch], errors="coerce"), "Phase": pd.to_numeric(df[ph], errors="coerce")})
    out["DeviceId"] = pd.to_numeric(df[dev], errors="coerce") if dev else (devices[0] if len(devices) == 1 else np.nan)
    if fn is not None:
        adv = df[fn].astype(str).str.contains("advance", case=False, na=False)
        out = out[adv]
        print(f"kept {adv.sum()} rows whose '{fn}' mentions Advance")
    out["Function"] = "Advance"
    return out.dropna().astype({"Parameter": "int64", "Phase": "int64", "DeviceId": "int64"}).reset_index(drop=True)

def config_from_text(text, devices):
    pairs = [p for p in re.split(r"[,\s;]+", text.strip()) if p]
    rows = []
    for p in pairs:
        chn, phs = re.split(r"[:=>-]+", p)
        for dev in devices:
            rows.append({"DeviceId": dev, "Phase": int(phs), "Parameter": int(chn), "Function": "Advance"})
    return pd.DataFrame(rows)

def check_events(ev):
    n1, n8, n82 = (ev.EventId == 1).sum(), (ev.EventId == 8).sum(), (ev.EventId == 82).sum()
    print(f"{len(ev):,} events | {ev.TimeStamp.min()} -> {ev.TimeStamp.max()} | devices: {list(ev.DeviceId.unique())}")
    print(f"green begins (1): {n1:,} | yellow begins (8): {n8:,} | detector on (82): {n82:,}")
    if n1 == 0 or n8 == 0: raise ValueError("No phase green/yellow events (codes 1 and 8) — cannot build green intervals.")
    if n82 == 0: raise ValueError("No detector-on events (code 82) — nothing to count arrivals with.")
    if abs(n1 - n8) > 0.02 * max(n1, n8):
        print(f"WARNING: green/yellow counts differ by {abs(n1-n8)}; some phase events may be missing, "
              "and bins with gaps will have a biased green ratio.")
    print("phases with greens:", sorted(ev.loc[ev.EventId == 1, "Parameter"].unique().tolist()))
    print("detector channels seen:", sorted(ev.loc[ev.EventId == 82, "Parameter"].unique().tolist()))

print("helpers ready")

In [ ]:
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Select your hi-res event file (CSV / Parquet / Excel), or cancel to use the bundled sample:")
up = files.upload() if IN_COLAB else {}
event_path = next(iter(up), None)

if event_path:
    events = standardise_events(read_any(event_path))
else:
    print("No file chosen: using the bundled sample.")
    events = sample_data.data.df(); event_path = "sample"
check_events(events)
devices = list(events.DeviceId.unique())

### Not sure which channels are the advance detectors?

The hi-res log carries three clues. **On-time** (82 on → 81 off): a 6-ft count loop is occupied for a fraction of a second,
a stop-bar presence loop holds a queued car for the whole red. **On-green share**: a stop-bar count loop only sees vehicles
crossing on green (share near 100 %), an advance loop sees arrivals on red too (share near the phase's green ratio).
**Phase called**: the phase whose call (event 43) registers within a quarter second of the detector-on.

The phase mapping from calls is reliable. The type guess is a *screening aid*: on a congested approach the queue can
back up over an advance loop and make it look like a presence loop, so read the table and decide, then paste the
suggested string into the settings below (or correct it). When in doubt, the signal's configuration page is the truth.

In [ ]:
def inspect_detectors(ev, call_window_s=0.25):
    # Per-channel on-time, on-green share and phase-called statistics, with a type guess.
    ev = ev.sort_values("TimeStamp").reset_index(drop=True)
    span = (ev.TimeStamp.max() - ev.TimeStamp.min()).total_seconds()
    greens, gratio = {}, {}
    for ph, e in ev[ev.EventId.isin([1, 8])].groupby("Parameter"):
        e = e.sort_values("TimeStamp")
        st, en = e.loc[e.EventId == 1, "TimeStamp"].values, e.loc[e.EventId == 8, "TimeStamp"].values
        if len(st) == 0 or len(en) == 0: continue
        greens[ph] = (st, en)
        i = np.searchsorted(en, st, side="left"); ok = i < len(en)
        gratio[ph] = ((en[i[ok]] - st[ok]).astype("timedelta64[ms]").astype(float) / 1000).sum() / span
    calls = ev[ev.EventId == 43]
    rows = []
    for ch, d in ev[ev.EventId.isin([81, 82])].groupby("Parameter"):
        on, off = d[d.EventId == 82], d[d.EventId == 81]
        if len(on) < 5: continue
        j = np.searchsorted(off.TimeStamp.values, on.TimeStamp.values, side="right"); ok = j < len(off)
        dur = (off.TimeStamp.values[j[ok]] - on.TimeStamp.values[ok]).astype("timedelta64[ms]").astype(float) / 1000
        k = np.searchsorted(calls.TimeStamp.values, on.TimeStamp.values, side="left"); okc = k < len(calls)
        dt = (calls.TimeStamp.values[k[okc]] - on.TimeStamp.values[okc]).astype("timedelta64[ms]").astype(float) / 1000
        called = calls.Parameter.values[k[okc]][dt <= call_window_s]
        ph = int(pd.Series(called).mode().iat[0]) if len(called) else None
        share = np.nan
        if ph in greens:
            st, en = greens[ph]
            i = np.searchsorted(st, on.TimeStamp.values, side="right") - 1; okg = i >= 0
            s0 = st[np.clip(i, 0, len(st) - 1)]; ni = np.searchsorted(en, s0, side="left")
            ny = np.where(ni < len(en), en[np.clip(ni, 0, max(len(en) - 1, 0))], np.datetime64("2100-01-01"))
            share = float(np.mean(okg & (on.TimeStamp.values < ny)))
        med, long_share = (np.median(dur), float(np.mean(dur > 3))) if len(dur) else (np.nan, np.nan)
        if long_share > 0.15 or med > 1.0: guess = "stop-bar presence"
        elif ph is not None and share > 0.9 and share > gratio.get(ph, 0) + 0.15: guess = "stop-bar count"
        else: guess = "advance?"
        rows.append(dict(channel=ch, actuations=len(on), median_on_s=med, p90_on_s=np.percentile(dur, 90) if len(dur) else np.nan,
                         long_share=long_share, phase_called=ph, on_green_share=share,
                         phase_green_ratio=gratio.get(ph, np.nan), guess=guess))
    t = pd.DataFrame(rows).set_index("channel").sort_values(["guess", "phase_called"])
    sugg = ", ".join(f"{ch}:{int(r.phase_called)}" for ch, r in t.iterrows() if r.guess == "advance?" and r.phase_called is not None)
    print("suggested advance_detectors =", repr(sugg) if sugg else "(none found)")
    return t.round(2)

inspect_detectors(events if "events" in dir() else sample_data.data.df())

### Build the advance-detector configuration

Fill in the settings form (or tick `use_config_file` to upload a detector table), then run the next cell.

In [ ]:
#@title Settings  { run: "auto" }
advance_detectors = "7:2, 21:6, 22:6"  #@param {type:"string"}
bin_size = 15                           #@param {type:"integer"}
latency_offset_seconds = 0              #@param {type:"number"}
use_config_file = False                 #@param {type:"boolean"}
#@markdown `advance_detectors` is `channel:phase` pairs, ignored when `use_config_file` is ticked.

In [ ]:
if event_path == "sample":
    config = sample_data.config.df()
elif use_config_file:
    print("Now select your detector configuration file:")
    up2 = files.upload() if IN_COLAB else {}
    cfg_path = next(iter(up2), None)
    if not cfg_path: raise ValueError("use_config_file is ticked but no file was chosen.")
    config = standardise_config(read_any(cfg_path), devices)
else:
    config = config_from_text(advance_detectors, devices)

seen = set(events.loc[events.EventId == 82, "Parameter"])
unseen = sorted(set(config.Parameter) - seen)
if unseen: print(f"WARNING: advance channels {unseen} never fired in this file — check the mapping.")
print("advance detector config:"); display(config)

In [ ]:
rp_mine = run_platoon_ratio(events, config, bin_size=bin_size,
                            latency_offset_seconds=latency_offset_seconds, output_dir="my_output")
stem = os.path.splitext(os.path.basename(event_path))[0]
out_csv = f"platoon_ratio_{stem}.csv"
rp_mine.to_csv(out_csv, index=False)
print(f"{len(rp_mine)} phase-bins written to {out_csv}")

print("\nPeriod summary (platoon ratio from summed parts, plus the naive mean for comparison):")
display(summarise(rp_mine, bin_size=bin_size))
print("\nArrival-type distribution by phase (count of bins):")
display(pd.crosstab([rp_mine.DeviceId, rp_mine.Phase], rp_mine.Arrival_Type).reindex(columns=range(1, 7), fill_value=0))
plot_rp(rp_mine, f"{stem}: platoon ratio by phase", bin_size=bin_size)
rp_mine.head(20)

In [ ]:
# Download the results (Colab only)
if IN_COLAB:
    files.download(out_csv)

## Notes on the implementation

- Green intervals are built per phase from event 1 (begin green) to the next event 8 (begin yellow), split across
  bin boundaries so a 15-minute bin only counts the green seconds that fall inside it.
- A green still open at the end of the data is extended to the end of its bin; a yellow with no preceding
  green (data starts mid-green) is assumed green from the start of its bin. These rules make the result
  identical whether the data is processed in one pass or incrementally in 15-minute chunks.
- Bins with zero green time for a phase are dropped rather than divided by zero.
- Missing phase events (a dropped green or yellow) bias the green ratio for that bin; the `check_events` helper
  warns when green and yellow counts disagree.
- Source: [`src/atspm/queries/platoon_ratio.sql`](https://github.com/ShawnStrasser/atspm/blob/main/src/atspm/queries/platoon_ratio.sql)
